In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.ensemble import GradientBoostingRegressor

# Import Bayesian Search from scikit-optimize
from skopt import BayesSearchCV
from skopt.space import Real, Integer






In [12]:
# ==========================================
# 1. SELECT PARAMETER (FULLY DYNAMIC)
# Options: 'iddq_uA' | 'leakage_current_nA' | 'propagation_delay_ns'
# ==========================================
PARAM_PREFIX = 'leakage_current_nA'

col_0h = f'{PARAM_PREFIX}_0h'
col_24h = f'{PARAM_PREFIX}_24h'
col_168h = f'{PARAM_PREFIX}_168h'

df = pd.read_csv('isro_burnin_ess_synthetic_dataset.csv')

# Drop NaNs specific to active parameter features & target
clean_df = df.dropna(subset=[col_0h, col_24h, col_168h]).copy()

# Feature engineering
clean_df['drift_0_to_24'] = clean_df[col_24h] - clean_df[col_0h]
clean_df['ratio_24_to_0'] = clean_df[col_24h] / (clean_df[col_0h] + 1e-6)

X = clean_df[[col_0h, col_24h, 'drift_0_to_24', 'ratio_24_to_0']]
y = clean_df[col_168h]

# Log transformation on target (no data leakage)
y_log = np.log1p(y)

X_train, X_test, y_train_log, y_test_log = train_test_split(
    X, y_log, test_size=0.2, random_state=42
)


In [13]:
# ==========================================
# 2. BAYESIAN OPTIMIZATION SEARCH SPACE
# ==========================================
search_space = {
    'n_estimators': Integer(50, 300),
    'max_depth': Integer(3, 8),
    'learning_rate': Real(0.01, 0.2, prior='log-uniform'),
    'subsample': Real(0.6, 1.0)
}

opt = BayesSearchCV(
    estimator=GradientBoostingRegressor(random_state=42),
    search_spaces=search_space,
    n_iter=15,  # Number of Bayesian optimization iterations
    cv=3,
    scoring='neg_mean_squared_error',
    random_state=42,
    n_jobs=-1
)

# Fit Bayesian Search
opt.fit(X_train, y_train_log)

# Best estimator found by Bayesian optimization
best_model = opt.best_estimator_

In [14]:
# ==========================================
# 3. EVALUATION & DRIFT PREDICTION
# ==========================================
y_pred_log = best_model.predict(X_test)
y_pred = np.expm1(y_pred_log)
y_test = np.expm1(y_test_log)

r2_log = r2_score(y_test_log, y_pred_log)

# Metric splits
overall_mae = mean_absolute_error(y_test, y_pred)
normal_mask = y_test < 100
normal_mae = mean_absolute_error(y_test[normal_mask], y_pred[normal_mask])

print(f"=== Bayesian Optimization Results ({PARAM_PREFIX}) ===")
print(f"Best Hyperparameters: {opt.best_params_}")
print(f"Log Scale R^2       : {r2_log:.4f}")
print(f"Normal Parts MAE    : {normal_mae:.4f}")
print(f"Overall MAE         : {overall_mae:.4f}")

=== Bayesian Optimization Results (leakage_current_nA) ===
Best Hyperparameters: OrderedDict({'learning_rate': 0.010035695368331576, 'max_depth': 4, 'n_estimators': 245, 'subsample': 0.6021690489905417})
Log Scale R^2       : 0.9434
Normal Parts MAE    : 4.0633
Overall MAE         : 2867.5476


In [15]:
# ==========================================
# 4. CONSTRUCT & PRINT PREDICTION TABLE
# ==========================================
# Construct DataFrame attached to original test index
results_df = X_test.copy()
results_df['Actual_168h'] = y_test
results_df['Predicted_168h'] = y_pred
results_df['Absolute_Error'] = (results_df['Actual_168h'] - results_df['Predicted_168h']).abs()

# Define table columns to display
table_cols = [col_0h, col_24h, 'Actual_168h', 'Predicted_168h', 'Absolute_Error']

print(f"=== Predictions Table ({PARAM_PREFIX}) ===")
print(results_df[table_cols].head(20).to_string())

=== Predictions Table (leakage_current_nA) ===
      leakage_current_nA_0h  leakage_current_nA_24h  Actual_168h  Predicted_168h  Absolute_Error
6252               237.4047                256.9385     257.6792      228.402166       29.277034
4684               162.1166                151.6076     183.5677      166.230277       17.337423
1731                57.5321                 59.1938      62.9080       63.024588        0.116588
4742                28.8282                 40.5816     248.4863      378.919156      130.432856
4521               143.0997                109.8100     150.7089      136.772411       13.936489
6340                62.5027                 59.6437      70.8238       67.771182        3.052618
576                 40.7711                 39.7629      42.7119       45.914561        3.202661
5202                29.5309                 32.1409      30.0210       35.634514        5.613514
6363               105.9254                105.1988     106.3495      112.191215